# Fisher basis for the per-MOSA noise model

Builds the per-MOSA basis covariances of the asymmetric XYZ noise model (`XYZSensitivityBackend`, TDI2,
averaged closed-form TFs, filter response) on the mojito_lite orbits and a one-week FD grid
(1e-4 - 0.1 Hz), with the `psd_multigpu_settings.py` preprocessing.

The covariance is linear in the per-MOSA PSD levels $P_m = b_m^2$, $C = \sum_m P_m B_m$, so unit-amplitude
evaluations give the exact derivatives. Saves `f`, `Boms`, `Btm` (6, 3, 3, Nf) and the covariance `C` at the
injection to `FISHER_DIR/basis_week<week>.npz`, read by `fisher_tm_analysis.ipynb`.

In [1]:
import os
import numpy as np

from lisatools.detector import L1Orbits
from lisatools.domains import FDSettings
from lisatools.globalfit.preprocessing import L1ProcessingStep
from lisatools.sensitivity import XYZSensitivityBackend
from lisatools.utils.utility import windowfun

L1_FOLDER = "/mnt/wd_hdd_6TB/nikos/DATA/global_fit/mojito_lite/"
FISHER_DIR = "/mnt/wd_hdd_6TB/nikos/DATA/global_fit/fisher_tm/"
WEEKS = [0]          # one-week segments after the 2% trim; e.g. [0, 40, 90] to check the epoch dependence
S_OMS, S_TM = 15e-12, 3e-15
BACKEND = "cpu"
os.makedirs(FISHER_DIR, exist_ok=True)

In [2]:
def basis_for_week(week):
    proc = L1ProcessingStep(L1_folder=L1_FOLDER, source_types=["noise"], verbose=False, do_plots=False,
                            orbits_class=L1Orbits, orbits_kwargs=dict(force_backend=BACKEND, frame="icrs"))
    trim = (dict(duration=0.02 + week * 7 / 731, is_percent=True, trimming_type="discard_from_start")
            if week else dict(duration=0.02, is_percent=True, trimming_type="from_each_end"))
    times, _ = proc.process(
        highpass_kwargs=dict(cutoff=5e-6, order=2, zero_phase=True),
        lowpass_kwargs=dict(cutoff=0.16, order=2, zero_phase=True),
        trim_kwargs=trim,
        downsample_kwargs=dict(target_fs=1 / 5.0, window=("kaiser", 5.0)),
        Tobs=7 * 86400.0,
    )
    fd = FDSettings.make_factory(min_freq=1e-4, max_freq=1e-1)(times=times, dt=proc.dt, force_backend=BACKEND)
    w, _ = windowfun("tukey", len(times), alpha=1e4 / (len(times) * proc.dt))
    _, orbits = proc.pour(settings=fd, window=w, return_orbits=True)
    f = np.asarray(fd.f_arr)
    be = XYZSensitivityBackend(orbits=orbits, settings=fd, force_backend=BACKEND, window_values=w,
                               filters_response=proc.get_total_response(f), noise_symmetry="asymmetric",
                               tdi_generation=2, mask_percentage=0.0, average_transfer_functions=True)
    I6, Z6 = np.eye(6), np.zeros((6, 6))
    B_oms = np.asarray(be.compute_sensitivity_matrix(I6, Z6)).reshape(6, 3, 3, -1)
    B_tm = np.asarray(be.compute_sensitivity_matrix(Z6, I6)).reshape(6, 3, 3, -1)
    C = np.asarray(be.compute_sensitivity_matrix(np.full(6, S_OMS), np.full(6, S_TM))).reshape(3, 3, -1)
    lin = S_OMS**2 * B_oms.sum(0) + S_TM**2 * B_tm.sum(0)
    print(f"week {week}: t0 = {times[0]:.1f} s, Nf = {f.size}, "
          f"linearity max rel err = {np.max(np.abs(lin - C)) / np.max(np.abs(C)):.1e}")
    np.savez(os.path.join(FISHER_DIR, f"basis_week{week}.npz"), f=f, Boms=B_oms, Btm=B_tm, C=C, df=fd.df)

for week in WEEKS:
    basis_for_week(week)

[load_data] L1Orbits(...) (incl. _setup) took 16.69s
[load_data] orbits._ensure_configured() took 18.95s


2026-10-06 11:01:07,795 - lisatools.globalfit.preprocessing - INFO - Initialized orbits from NOISE file.


[load_data] NOISE f.tdis.xyz_doppler[:] took 5.75s shape=(25246480, 3)


/home/karnesis/anaconda3/envs/erebor_env/lib/python3.12/site-packages/cupyx/jit/_interface.py:247: FutureWarning: cupyx.jit.rawkernel is experimental. The interface can change in the future.
  cupy._util.experimental('cupyx.jit.rawkernel')


week 0: t0 = 98992259.8 s, Nf = 60420, linearity max rel err = 4.5e-16
